In [ ]:
import scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered

import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import pandas_datareader as web

In [ ]:
AAPL_dates = pd.DataFrame(data=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/EventStudy/PKL/Stocks/AAPL.pkl"))
spx = pd.DataFrame(data=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl"))
AAPL = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/Data/PKL/Stocks/AAPL.pkl")

In [ ]:
spx["date"] = pd.to_datetime(spx.index)
AAPL["date"] = pd.to_datetime(AAPL["date"])

In [ ]:
df_AAPL = pd.DataFrame(data=AAPL[["date","adjClose"]]).rename(columns={0:"adjClose"})
df_spx =  pd.DataFrame(data=spx[["date","adjclose"]]).rename(columns={0:"adjClose"})

In [ ]:
df_AAPL["adjClose(t-1)"]=df_AAPL["adjClose"].shift(periods=1)
df_AAPL["Stock returns"]=np.log(df_AAPL["adjClose"]/df_AAPL["adjClose(t-1)"])

df_spx["adjclose(t-1)"]=df_spx["adjclose"].shift(periods=1)
df_spx["SPX returns"]=np.log(df_spx["adjclose"]/df_spx["adjclose(t-1)"])

In [ ]:
df = pd.merge(df_AAPL,df_spx,on="date")
df = df[["date","Stock returns","SPX returns"]]

In [ ]:
# df.sort_values(by="date",inplace=True)

In [ ]:
df_dates = AAPL_dates

In [ ]:
# df_dates.date.to
# date_time = now.strftime("%m/%d/%Y, %H:%M:%S")
df_dates["date"] = [date_obj .strftime('%Y-%m-%d') for date_obj  in df_dates.date]
df["date"] = [date_obj .strftime('%Y-%m-%d') for date_obj  in df.date]

In [ ]:
df_dates["date"]

In [ ]:
# set(df_dates["date"]).intersection(set(df["date"]))

In [ ]:
# df_dates["date"] = pd.DataFrame(data=set(df_dates["date"]).intersection(set(df["date"])))

In [ ]:
events_db = []
for date, row in df_dates.T.iteritems():
    print(row["date"])
    new_row = {'event_date': np.datetime64(row["date"]),'market_ticker':'SPX returns','security_ticker': 'Stock returns'}
    events_db.append(new_row)

In [ ]:
returns = df

In [ ]:
len(df_dates)

In [ ]:
len(events_db)

In [ ]:
returns

In [ ]:
returns.sort_values(by="date")
returns.index = returns.date

In [ ]:
returns.to_csv("../CSV/returns_list.csv")

In [ ]:
es.Single.import_returns("../CSV/returns_list.csv")
# es.Single.import_FamaFrench("../CSV/FamaFrenchSPY.csv")

In [ ]:
events_db

In [ ]:
# tickers = ['SPY']
releases = dict()
ticker= "Stock returns"
events = [event for event in events_db if event['security_ticker']==ticker]
releases = es.Multiple.from_list(
    events,
    es.Single.market_model,#FamaFrench_3factor,
    event_window= (-5,5)
)

In [ ]:
releases.plot(AAR=True,CI=True)
    # plt.figure(figsize=(20, 20))
plt.xlabel('Event Day')
plt.title('Event Study - Daily Frequency\nS&P500 return and VIX Return')

In [ ]:
releases.AAR

In [ ]:
list(range(-5, 6))

In [ ]:
# Compute the cumulative abnormal returns for the event window
cumulative_abnormal_returns = releases.CAAR#event_abnormal_returns.cumsum()
# Plot the event study
plt.figure(figsize=(15,6))
plt.plot(list(range(-5, 6)),cumulative_abnormal_returns)
plt.axhline(y=0, color='black', linestyle='--')
plt.xlabel('Date')
plt.ylabel('Cumulative Abnormal Returns')
plt.title('Sentiment Cumulative Abnormal Returns')
plt.show()